AWL Human-in-the-Loop Workflow Demo - [Open this notebook in JupyterLite (async variant)](https://repolab.github.io/jupyterlite-playground/lab/index.html?fromURL=https://raw.githubusercontent.com/OO-LD/awl-python/refs/heads/main/examples/human_in_the_loop_async.ipynb)

Note: The sync code in this notebook does not run in a pyodide/jupyterlite environment, please make use of the [async variant](https://github.com/OO-LD/awl-python/blob/main/examples/human_in_the_loop_async.ipynb) in this cases.

Install required packages

In [ ]:
%pip install bokeh==3.7.3 jupyter_bokeh panel==1.7.5 awl[hitl]

Define data models and a workflow. Functions / steps decorated with hitl (human in the loop) will prompt the user for input by an auto-generated form (see also [OO-LD UI Demo](https://github.com/OO-LD/oold-python/blob/main/examples/linked_data_editor.ipynb))

In [ ]:
from enum import Enum
from typing import Any

from oold.model import LinkedBaseModel
from pydantic import Field

from awl.legacy.hitl import entry_point, hitl


class MachineParams(LinkedBaseModel):
    """Parameters transferred to the machine"""

    model_config = {
        "json_schema_extra": {
            "required": ["param1"],
        },
    }
    param1: int = Field(50, ge=0, le=100)


class Quality(str, Enum):
    good = "Good"
    bad = "Bad"
    worse = "Worse"


class ProcessDocumentation(LinkedBaseModel):
    """Visual result inspection"""

    quality: Quality
    """Good is defined as..."""


class YesNoEnum(Enum):
    yes = True
    no = False


class YesNo(LinkedBaseModel):
    answer: bool  # YesNoEnum


@hitl
def set_machine_params(params: MachineParams):
    # transfer params to machine
    return params


@hitl
def document_result(params: ProcessDocumentation):
    # validate documentation
    return params


@hitl
def document_more(params: YesNo) -> YesNo:
    return params


def archive_data(params: Any):
    # store documentation in database
    pass


@entry_point(gui=True, jupyter=True)
def workflow():
    print("Enter")
    machine_params = set_machine_params()  # prompts user
    do_document_more = True
    while do_document_more:
        result_evaluation = document_result()  # prompts user
        do_document_more = (document_more()).answer

    print("Machine parameters: ", machine_params)
    print("Result evaluation: ", result_evaluation)
    archive_data(machine_params)  # runs automatically
    archive_data(result_evaluation)  # runs automatically


workflow()
print("Done")